# ClimateWeaver
## From forcing to coherent daily weather

**Learn the distributions. Keep the weather together.**

ClimateWeaver demonstrates a compact climate-emulation recipe: fit monthly conditional quantiles against forcing, borrow a real historical weather year and map that shared template into a target distribution. Using the same template days across variables preserves their day pairing while the marginal distributions change.

This notebook contributes an explicit CPU-friendly quantile-ridge core, a real historical held-year walkthrough, seven-variable physical checks and distribution plots. The example uses a small historical block to explain the mechanics. It is not a full-resolution climate-model benchmark or an official index score.

[Source and companion files](https://github.com/GISWLH/AImodel-EarthScience/tree/main/kaggle-geoscience/climateweaver) · Author: **giswlh / Longhao Wang**, with AI assistance.


In [ ]:
import os, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
plt.rcParams.update({'figure.dpi':115,'axes.spines.top':False,'axes.spines.right':False,'axes.titleweight':'bold'})
OUTPUT=Path('/kaggle/working') if Path('/kaggle').exists() else Path.cwd()/'outputs'
OUTPUT.mkdir(parents=True,exist_ok=True)


## The method in three steps

**Conditional distribution:** for each grid cell, calendar month and quantile, fit a regularized linear response to a forcing design. The example uses an intercept and training-standardized annual CO2.

**Coherent weather:** map a training year's daily values into quantile coordinates, then map those coordinates through the target-year quantiles. The same real year is used for all seven variables. Precipitation uses a cube-root transform to reduce skewness; inversion produces nonnegative rain.

**Physical checks:** generate `tas`, `tasmax`, `tasmin`, `pr`, `huss`, `psl` and `sfcWind`, enforce temperature ordering, and check finite/nonnegative outputs. A full competition pipeline would then compute the official indices with fixed historical thresholds. Those index maps are not fitted directly here.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
CLIMX_VARIABLES=['tas','tasmax','tasmin','pr','huss','psl','sfcWind']
CLIMX_PROBS=np.array([.01,.02,.035,.05,.075,.1,.15,.2,.25,.3,.35,.4,.45,.5,.55,.6,.65,.7,.75,.8,.85,.9,.925,.95,.965,.98,.99])

def climx_month_windows():
    centers=np.round((np.arange(12)+.5)*365/12-.5).astype(int)
    return (centers[:,None]+np.arange(-30,31)[None,:])%365

def climx_fit_quantile_core(design,daily,ridge=None,backend='numpy'):
    """Daily arrays(year,365,cell). Fit each month/quantile/cell on forcing.

    numpy is the default. torch_cuda explicitly uses an already-authorized GPU.
    This core omits the full pipeline's fitted forcing memory/aerosol PCA,
    deseasonalization, calibrated tails, weather ensemble and official indices.
    """
    design=np.asarray(design,dtype=np.float64);daily=np.asarray(daily,dtype=np.float32)
    assert daily.shape[:2]==(len(design),365) and np.isfinite(daily).all()
    windows=climx_month_windows()
    q=np.quantile(daily[:,windows,:],CLIMX_PROBS,axis=2).transpose(1,2,0,3)
    penalties=np.r_[0.,np.ones(design.shape[1]-1)] if ridge is None else np.asarray(ridge)
    a=design.T@design+np.diag(penalties);b=design.T@q.reshape(len(design),-1)
    if backend=='torch_cuda':
        import torch
        assert torch.cuda.is_available()
        with torch.inference_mode():coef=torch.linalg.solve(torch.as_tensor(a,device='cuda'),torch.as_tensor(b,device='cuda')).cpu().numpy()
    else:
        assert backend=='numpy';coef=np.linalg.solve(a,b)
    return np.ascontiguousarray(coef.reshape(design.shape[1],12,len(CLIMX_PROBS),daily.shape[-1]))

def climx_daily_knots(coefficients,x,precipitation=False):
    monthly=np.sort(np.einsum('p,pmkc->mkc',np.asarray(x),coefficients,optimize=False),axis=1)
    if precipitation:monthly=np.maximum(monthly,0)
    position=(np.arange(365)+.5)*12/365-.5;left=np.floor(position).astype(int);w=(position-left)[:,None,None]
    return monthly[left%12]*(1-w)+monthly[(left+1)%12]*w

def climx_map_weather(template,source_knots,target_knots):
    """Same real template days/cells across7variables preserve joint day pairing.

    This bounded reference kernel interpolates on fixed quantile probabilities;
    it clips probabilities to[0,1] and is not the production tail extrapolator.
    """
    output=np.empty_like(template,dtype=np.float32)
    for d in range(len(template)):
        for c in range(template.shape[1]):
            p=np.interp(template[d,c],source_knots[d,:,c],CLIMX_PROBS)
            output[d,c]=np.interp(p,CLIMX_PROBS,target_knots[d,:,c])
    return output

def climx_historical_core(data_root,backend='numpy',return_fields=False):
    """Real4year/4cell mechanics check; year4held from fitting. No rnNSE claim."""
    import xarray as xr
    files=list(Path(data_root).rglob('*hist_train_1850_1931.nc'))
    assert len(files)==1,'Attach competition historical lite NetCDF data'
    with xr.open_dataset(files[0],decode_times=False) as ds:
        assert ds.time.attrs['calendar'] in ('noleap','365_day')
        assert np.array_equal(ds.time.values[:4*365],np.arange(4*365)), 'Expected complete daily years'
        co2=np.asarray(ds.CO2_LBC.isel(forcing_scenario=0).values[:4],dtype=np.float64)
        center=co2[:3].mean();scale=max(float(co2[:3].std()),1e-12);design=np.c_[np.ones(4),(co2-center)/scale]
        raw={}
        for v in CLIMX_VARIABLES:
            arr=np.asarray(ds[v].isel(forcing_scenario=0,time=slice(0,4*365),lat=6,lon=slice(0,4)).values,dtype=np.float32)
            if v=='pr':
                assert ds[v].attrs.get('units')=='kg m-2 s-1';arr*=86400.
            raw[v]=arr.reshape(4,365,4)
    generated={};diagnostics=[]
    for v,arr in raw.items():
        transformed=np.cbrt(np.maximum(arr,0)) if v=='pr' else arr
        coef=climx_fit_quantile_core(design[:3],transformed[:3],backend=backend)
        # The template is year3, always inside the training partition.
        source=climx_daily_knots(coef,design[2],v=='pr');target=climx_daily_knots(coef,design[3],v=='pr')
        z=climx_map_weather(transformed[2],source,target)
        generated[v]=np.maximum(z,0)**3 if v=='pr' else z
        if v in ['pr','huss','sfcWind']:generated[v]=np.maximum(generated[v],0)
    generated['tasmax']=np.maximum(generated['tasmax'],generated['tas']);generated['tasmin']=np.minimum(generated['tasmin'],generated['tas'])
    for v,p in generated.items():
        assert np.isfinite(p).all()
        diagnostics.append({'variable':v,'historical_hold_year':1853,'daily_MAE_in_variable_units':float(np.mean(abs(p-raw[v][3]))),'minimum_generated':float(p.min()),'maximum_generated':float(p.max())})
    report={'kind':'limited_real_historical_mechanics_check','official_rnNSE':None,'representative_validation':False,'test_targets_read':False,'train_years':[1850,1851,1852],'held_year':1853,'cells':4,'backend':backend,'all7daily_fields_generated':True,'diagnostics':diagnostics}

    if return_fields:return report,generated,{v:a[3] for v,a in raw.items()}
    return report


## A real historical holdout

Train on 1850–1852 and generate 1853 at four grid cells using 1852, the last training year, as the shared weather template. The held 1853 daily observations are used only for the diagnostic comparisons below. CO2normalization is fitted on the training years. The example reads the historical lite NetCDF; it never reads future target labels.

A single borrowed weather realization is not a day-by-day forecast. Compare seasonal behavior and distributions, and treat the daily errors as implementation diagnostics in each variable's own units.


In [ ]:
HIST_ROOT=os.environ.get('CLIMX_TRAIN_ROOT','/kaggle/input')
report,generated,observed=climx_historical_core(HIST_ROOT,backend='numpy',return_fields=True)
units={'tas':'K','tasmax':'K','tasmin':'K','pr':'mm/day','huss':'kg/kg','psl':'Pa','sfcWind':'m/s'}
checks=pd.DataFrame(report['diagnostics']);checks['units']=checks.variable.map(units)
display(checks[['variable','units','daily_MAE_in_variable_units','minimum_generated','maximum_generated']])
assert (generated['tasmin']<=generated['tas']).all() and (generated['tas']<=generated['tasmax']).all()
assert (generated['pr']>=0).all();print('Seven finite daily fields; temperature ordering and nonnegative rain verified.')
(OUTPUT/'climateweaver_checks.json').write_text(json.dumps(report,indent=2))


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,3.8))
for variable,ax,label in [('tas',axes[0],'Temperature (K)'),('pr',axes[1],'Precipitation (mm/day)')]:
    for data,name,color in [(observed,'Observed held year','#2274a5'),(generated,'Mapped historical weather','#ef8354')]:
        a=np.sort(data[variable].ravel());p=(np.arange(len(a))+.5)/len(a);ax.plot(a,p,label=name,color=color)
    ax.set(xlabel=label,ylabel='Empirical cumulative probability',title=f'{variable}: compare distributions');ax.grid(alpha=.2)
axes[-1].legend(fontsize=8);plt.tight_layout();plt.show()


## Keep temperature relationships physical

The generator enforces minimum≤mean≤maximumtemperature on every day and cell. The plot below shows one generated grid cell; it illustrates the constraint, not deterministic prediction of the observed weather sequence.


In [ ]:
fig,ax=plt.subplots(figsize=(11,3.5));days=np.arange(365)
ax.fill_between(days,generated['tasmin'][:,0],generated['tasmax'][:,0],color='#2274a5',alpha=.22,label='Generated daily min–max')
ax.plot(days,generated['tas'][:,0],color='#2274a5',lw=1,label='Generated daily mean')
ax.set(xlabel='Day of year',ylabel='Temperature (K)',title='One shared weather template, physically ordered temperatures');ax.legend();plt.tight_layout();plt.show()


## Where to take it next

For serious cross-scenario evaluation, refit the forcing and distribution models with whole scenarios held out. Add physically motivated forcing memory and aerosol predictors, a better seasonal baseline, calibrated tail coordinates and several coherent weather members. Check precipitation extremes and spell persistence separately; a smooth-looking temperature distribution does not establish skill for heavy rain.

The core here clips quantile probabilities to the interpolation range and uses a small CO2design. Those are explicit teaching choices, not the complete production emulator. Computing official indices requires the competition's historical thresholds and scoring definitions.

**Credit:** the distributional-emulation recipe is inspired by [maverickss26's ClimX notebook](https://www.kaggle.com/code/maverickss26/climx-sota-main-track). The implementation here is a compact explanatory core. Competition data remain subject to their own terms; no future labels or private model weights are redistributed.
